# Jalon 3 · Moteur d’optimisation des livraisons

Le mode **lissé**, sélectionné par défaut, assemble des plans de couples avec des plafonds par usine et des profils stricts. Les modes **pilote** et **complet** relancent le moteur de génération des plans à bas coût, avec et sans imputation. Chaque mode écrit ses résultats dans `results/`.

[Guide du projet](../README.md) · [Annexe mathématique](modelisation_mathematique.md) · [Bilan et validation du jalon 4](../milestone4/bilan_avant_apres.ipynb)

## 1. Choisir le périmètre

Chaque ligne est entière et son poids et son volume décrivent la charge totale. Le jeu préparé vient du commit `a0e8aba`. Le lissage utilise les 4 389 lignes du périmètre avec imputation, commun aux plans candidats. Les autres modes traitent aussi le périmètre sans imputation chiffrable, qui contient 4 251 lignes.

In [1]:
import os
from pathlib import Path
import subprocess
import sys

import pandas as pd

mode = "lisse"  # "lisse", "pilote" ou "complet"
couple = "SUP_00112:PLT_004"

root = Path.cwd()
if root.name == "milestone3":
    root = root.parent
published = root / "milestone4" / "resultats_verifies"
package = "delivery_schedule_optimization"
assert (root / "src" / package / "optimisation_cout.py").is_file(), "Ouvrir le notebook depuis le dépôt ou le dossier milestone3."
assert mode in {"lisse", "pilote", "complet"}

output = root / "results" / f"moteur-notebook-{mode}"
workers = 2 if mode == "pilote" else min(7, os.cpu_count() or 1)
env = {
    **os.environ,
    "PYTHONPATH": str(root / "src"),
    "OMP_NUM_THREADS": "1",
    "OPENBLAS_NUM_THREADS": "1",
}
print(f"Mode {mode} : {couple if mode == 'pilote' else 'tous les couples'}")

Mode lisse : tous les couples


## 2. Calculer ou sélectionner les plans

En mode pilote ou complet, le solveur choisit le profil, les départs et les chargements. Il respecte la fenêtre J−7 à J, les jours ouvrés et les deux capacités. Il minimise le coût total camion par camion selon la grille poids-distance, puis calcule des bornes de coût.

En mode lissé, on sélectionne un plan par couple parmi les exports des branches figées. Chaque plan doit couvrir les mêmes lignes et respecter les jours du profil. Les plafonds quotidiens par usine viennent du plan d’Arianne. Le solveur minimise d’abord le coût, puis le pic global et enfin le nombre de camions à coût et pic égaux.

In [2]:
def execute(command):
    with subprocess.Popen(command, cwd=root, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="")
        if process.wait():
            raise RuntimeError("Le calcul a échoué. La sortie ci-dessus indique la cause.")

if mode == "lisse":
    execute([sys.executable, "-m", f"{package}.lissage_publie", "--output", str(output)])
else:
    calculation = output if mode == "pilote" else output / "calcul"
    command = [
        sys.executable, "-m", f"{package}.optimisation_cout", "--earliest", "2026-11-02", "--adaptive",
        "--tight-window-cuts", "--workers", str(workers), "--output", str(calculation),
    ]
    if mode == "pilote":
        command += ["--couple", couple, "--exact-size", "10", "--bound-seconds", "20"]
    else:
        command += ["--exact-size", "0", "--bound-seconds", "12",
                    "--incumbent", str(published / "affectations.csv")]

    execute(command)
    if mode == "complet":
        floors = output / "bornes_fenetres.csv"
        columns = ["scope", "supplier_id", "plant", "forced", "sans_calendrier", "fenetres_separees"]
        pd.read_csv(published / "couples.csv", usecols=columns).to_csv(floors, index=False)
        execute([
            sys.executable, "-m", f"{package}.retenir_meilleurs_couts",
            "--earliest", "2026-11-02", "--run", str(published), "--run", str(calculation),
            "--floors", str(floors), "--output", str(output),
        ])

{
  "scope": "avec_imputation",
  "lines": 4389,
  "couples": 180,
  "trucks": 755,
  "cost_eur": 109365.68,
  "reference_eur": 176724.61,
  "savings_eur": 67358.93,
  "extra_vs_prime_eur": 19.89,
  "global_peak": 43,
  "plant_peaks": {
    "PLT_001": 8,
    "PLT_002": 31,
    "PLT_003": 9,
    "PLT_004": 13
  },
  "plant_limits": {
    "PLT_001": 8,
    "PLT_002": 31,
    "PLT_003": 9,
    "PLT_004": 13
  },
  "eligible_plans": 1205,
  "source_commits": {
    "prime": "b492f6cdfb06c9e88ce164f6b54c1b36fe8aa479",
    "Arianne": "1ee10abcee724e74c4689a6f3124d15498dcb9fb",
    "Gilbert 3.1": "8d8f5e4874aa89757e81d5d137aa3a496bf9858e",
    "presentation": "f69b088706e654332df69fa8039b82b42c0c1910",
    "main": "13a8ab9ccc9e4a2a1caff98e9a5231222af773c7",
    "Wissam": "4c78cdbc2aebe6d1d79432874141c64ea4553ade",
    "Gilbert 3": "1542dc7c83c511aeee9081b1d4ce56a45cd113b2"
  }
}


## 3. Lire le résultat

Les fichiers exportent les profils, les camions et les affectations. Pour le lissage, le coût minimal est prouvé parmi les plans candidats validés. Les modes de génération affichent l’écart entre le meilleur plan et la borne inférieure du modèle par couple.

In [3]:
import json

report = "bilan.json" if mode == "pilote" else "comparatif.json"
result = json.loads((output / report).read_text(encoding="utf-8"))
if mode == "lisse":
    metrics = ["lines", "trucks", "cost_eur", "savings_eur", "extra_vs_prime_eur", "global_peak"]
    print(pd.DataFrame([result])[metrics].to_string(index=False))
    print("Pics par usine :", result["plant_peaks"])
else:
    summary = pd.DataFrame.from_dict(result, orient="index")
    metrics = ["lines", "trucks", "reference_eur", "cost_eur", "saving_eur", "gap_eur", "certified"]
    print(summary[metrics].to_string())

couples = pd.read_csv(output / "couples.csv", dtype={"profile": str})
print()
detail = "source" if mode == "lisse" else "certified"
print(couples[["scope", "supplier_id", "plant", "profile", "trucks", detail]].head(10).to_string(index=False))
print()
print("Aperçu des camions calculés")
trucks = pd.read_csv(output / "camions.csv")
print(trucks[["scope", "date", "truck", "lines", "weight", "volume", "cost_eur"]].head(6).round(2).to_string(index=False))
print()
for name in ("couples.csv", "camions.csv", "affectations.csv", report):
    path = output / name
    assert path.is_file(), name
    print(path.relative_to(root))

 lines  trucks  cost_eur  savings_eur  extra_vs_prime_eur  global_peak
  4389     755 109365.68     67358.93               19.89           43
Pics par usine : {'PLT_001': 8, 'PLT_002': 31, 'PLT_003': 9, 'PLT_004': 13}

          scope supplier_id   plant profile  trucks       source
avec_imputation   SUP_00001 PLT_004   1 2 3      18 presentation
avec_imputation   SUP_00003 PLT_001       1       4    Gilbert 3
avec_imputation   SUP_00005 PLT_001     3 4       6      Arianne
avec_imputation   SUP_00008 PLT_002       3       3    Gilbert 3
avec_imputation   SUP_00009 PLT_004       0       1      Arianne
avec_imputation   SUP_00010 PLT_001     1 4       3      Arianne
avec_imputation   SUP_00010 PLT_002       4       4  Gilbert 3.1
avec_imputation   SUP_00010 PLT_004     2 4       5         main
avec_imputation   SUP_00011 PLT_002     0 1       4      Arianne
avec_imputation   SUP_00011 PLT_004     0 4       6      Arianne

Aperçu des camions calculés
          scope       date   truck  l